# Laboratorium 1 - analiza koszykowa

## Przygotowanie

 * pobierz i wypakuj dataset: https://kaggle.com/datasets/rashikrahmanpritom/groceries-dataset-for-market-basket-analysismba?resource=download&select=basket.csv
   * alternatywnie, pobierz plik `basket.csv` z Teamsów
 * [opcjonalnie] Utwórz wirtualne środowisko
 `python3 -m venv ./recsyslab1`
 * zainstaluj potrzebne biblioteki:
 `pip install more-itertools`

## Część 1. - przygotowanie danych

In [ ]:
# importujemy wszystkie potrzebne pakiety

import random
from typing import NamedTuple
from more_itertools import powerset

In [ ]:
# definiujemy stałe

PATH = './basket.csv'
EPSILON = 0.001
N = 20          # liczba rekomendacji zwracanych przez recommend()
K = [3, 5, 10]  # wartości k używane przy ewaluacji
N_EVAL = 200    # liczba koszyków użytych do ewaluacji
SEED = 42

In [ ]:
# wczytujemy dane o koszykach

def read_baskets(path: str) -> list[set[str]]:
    with open(path) as f:
        raw = f.read()
    baskets = [
        {item.lower() for item in row.split(',') if item}
        for row in raw.split('\n')[1:]
        if row
    ]
    return baskets


def unique_products(baskets: list[set[str]]) -> list[str]:
    products: set[str] = set()
    for basket in baskets:
        products.update(basket)
    return sorted(products)


baskets = read_baskets(PATH)
products = unique_products(baskets)
print(f"Wczytano {len(baskets)} koszyków i {len(products)} unikalnych produktów.")

## Część 2. - obliczanie wskaźników

In [ ]:
# obliczamy strukturę danych (słownik) przechowującą wszystkie interesujące wartości `support`

Supports = dict[str, float]


def get_supports(
    baskets: list[set[str]],
    all_products: list[str],
    epsilon: float,
) -> Supports:
    raise NotImplementedError()


supports = get_supports(baskets, products, EPSILON)
supports

In [ ]:
# definiujemy funkcje obliczające support, confidence i lift

def support(supports: Supports, products: set[str]) -> float:
    raise NotImplementedError()


def confidence(
    supports: Supports,
    prior: set[str],
    following: set[str],
) -> float:
    raise NotImplementedError()


def lift(
    supports: Supports,
    prior: set[str],
    following: set[str],
) -> float:
    raise NotImplementedError()

In [ ]:
print(support(supports, {'whole milk', 'rolls/buns'}))
print(confidence(supports, {'whole milk', 'rolls/buns'}, {'yogurt'}))
print(lift(supports, {'whole milk', 'rolls/buns'}, {'yogurt'}))

## Część 3. - generowanie rekomendacji

In [ ]:
# wyznaczamy listę potencjalnych rekomendacji
# rekomendowane artykuły powinny mieć lift > 1 i jak najwyższe confidence

class Candidate(NamedTuple):
    item: str
    subbasket: set[str]
    confidence: float
    lift: float


def generate_next_product_candidates(
    basket: set[str],
    products: list[str],
    supports: Supports,
) -> list[Candidate]:
    """Zwraca listę kandydatów posortowanych malejąco po confidence."""
    raise NotImplementedError()

In [ ]:
# wybieramy top-n unikalnych produktów z listy kandydatów

def recommend(
    basket: set[str],
    products: list[str],
    supports: Supports,
    n: int,
) -> list[str]:
    """Zwraca top-n rekomendacji na podstawie analizy koszykowej."""
    raise NotImplementedError()

In [ ]:
print("Koszyk:", baskets[1])
print("Rekomendacje:", recommend(baskets[1], products, supports, N))

In [ ]:
print("Koszyk:", baskets[33])
print("Rekomendacje:", recommend(baskets[33], products, supports, N))

## Część 4. - ewaluacja rekomendacji: MRR@k

### Metryka

**MRR@k** (Mean Reciprocal Rank) mierzy, na jakiej *pozycji* pojawił się ukryty produkt wśród top-k rekomendacji. Nagradza wyższe miejsca na liście:

$$\text{MRR@k} = \frac{1}{|\mathcal{Q}|} \sum_{q \in \mathcal{Q}} \frac{1}{\text{rank}_q}$$

gdzie:
- $\mathcal{Q}$ to zbiór przypadków testowych
- $\text{rank}_q$ to pozycja pierwszego trafnego produktu w top-k dla danego przypadku testowego $q$ (lub $\infty$, jeśli nie znaleziono — wkład do sumy wynosi wtedy 0).

### Protokół ewaluacji

Dla każdego koszyka testowego:
1. losowo wydzielamy jeden ukryty produkt; ten produkt byłby tym, który użytkownik najchętniej umieści w koszyku i tym samym tym, który powinien zaproponować nasz algorytm,
2. pozostałe produkty stanowią kontekst (czyli koszyk) przekazywany rekomendatorowi,
3. sprawdzamy, na której pozycji top-k pojawia się ukryty produkt.

Porównamy dwa algorytmy:
- **basket** – rekomendator oparty na analizie koszykowej (confidence + lift),
- **random** – rekomendator losowy (baseline).

Porównanie możemy przeprowadzić dla różnych wartości $k$.

In [ ]:
# algorytm losowy - punkt odniesienia (baseline)

def recommend_random(
    basket: set[str],
    products: list[str],
    k: int,
    rng: random.Random | None = None,
) -> list[str]:
    """Zwraca k losowo wybranych produktów spoza koszyka."""
    raise NotImplementedError()

In [ ]:
# implementacja metryki MRR@k

def reciprocal_rank_at_k(recommended: list[str], relevant: set[str], k: int) -> float:
    """Zwraca odwrotność pozycji pierwszego trafnego produktu w top-k (0 jeśli brak trafienia)."""
    raise NotImplementedError()

In [ ]:
# podział koszyka na kontekst i ukryte produkty (ground truth)

def split_basket(
    basket: set[str],
    sample_size: int,
    rng: random.Random,
) -> tuple[set[str], set[str]]:
    """Losowo wydziela `sample_size` produktów jako cel dla rekomendacji; reszta to koszyk"""
    raise NotImplementedError()

In [ ]:
# typy reprezentujące wyniki ewaluacji

class MetricAtK(NamedTuple):
    k: int
    mrr: float


class EvaluationResults(NamedTuple):
    basket: MetricAtK
    random: MetricAtK

In [ ]:
# ewaluacja obu rekomendatorów dla zadanego k

def evaluate(
    baskets: list[set[str]],
    products: list[str],
    supports: Supports,
    k: int,
    n_eval: int = N_EVAL,
    seed: int = SEED,
) -> EvaluationResults:
    """
    Generuje `n_eval` przypadków testowych przy uzyciu funkcji `split_basket`.
    Dla kazdego przypadku testowego uruchamia oba algorytmy, a następnie oblicza `MMR@k`.
    Zwraca średnie wartości MMR@k dla wskazanego `k` i dla obu algorytmów.
    """
    raise NotImplementedError()

In [ ]:
# wyświetlamy wyniki i porównujemy algorytmy dla każdego k z listy K

def print_evaluation(results: EvaluationResults) -> None:
    col_w = 10
    header = f"{'Algorytm':<12} | {'mrr@' + str(results.basket.k):>{col_w}}"
    print(header)
    print('-' * len(header))
    for algo, metric in results._asdict().items():
        print(f"{algo:<12} | {metric.mrr:>{col_w}.4f}")


for k in K:
    print_evaluation(evaluate(baskets, products, supports, k))
    print()